# Analisi degli Outlier — Sprint 2

| US | Descrizione | Responsabile |
|---|---|---|
| US-10 | [Outlier - Metodo Deviazione Standard](#us-10) | Youssra |
| US-11 | [Outlier - Metodo Boxplot (IQR)](#us-11) | Youssra |
| US-12 | [Outlier - DBSCAN](#us-12) | Youssra |
| US-13 | [Outlier - Isolation Forest e Confronto](#us-13) | Youssra |

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

from src.preprocessing import detect_outliers_isolation_forest, detect_outliers_iqr, detect_outliers_std
from src.visualization import DEFAULT_DISPLAY_NAMES

df = pd.read_csv('../data/processed/data_clean.csv')
print(f'Dataset caricato: {df.shape[0]} righe, {df.shape[1]} colonne')

---

## US-10 · Analisi degli Outlier — Metodo Deviazione Standard <a id="us-10"></a>

**Responsabile**: Youssra Zarouky

Un valore e considerato outlier se si trova a piu di **3 deviazioni standard** dalla media della sua variabile.

In [ ]:
numerical_cols = [
    'previous_qualification_grade', 'admission_grade', 'age_at_enrollment',
    'cu_1st_sem_credited', 'cu_1st_sem_enrolled', 'cu_1st_sem_evaluations',
    'cu_1st_sem_approved', 'cu_1st_sem_grade', 'cu_1st_sem_without_evaluations',
    'cu_2nd_sem_credited', 'cu_2nd_sem_enrolled', 'cu_2nd_sem_evaluations',
    'cu_2nd_sem_approved', 'cu_2nd_sem_grade', 'cu_2nd_sem_without_evaluations',
    'unemployment_rate', 'inflation_rate', 'gdp'
]

outlier_summary = detect_outliers_std(df, numerical_cols)
print(outlier_summary.to_string(index=False))

### Osservazioni

**Variabili accademiche — lo zero è semantico, non estremo.** Le variabili `cu_*_approved` e `cu_*_grade` mostrano il maggior numero di outlier con questo metodo. Tuttavia, come già evidenziato nell'analisi delle distribuzioni (US-07), il valore `0` non rappresenta un dato estremo ma una **categoria semantica**: uno studente con 0 esami superati riceve `grade = 0` come placeholder convenzionale, non un voto reale nella fascia bassa. Il sistema portoghese non ammette voti tra 0.5 e 9, perciò lo zero è strutturalmente separato dai voti effettivi (10–20). Questi valori non sono errori né anomalie — sono la firma del dropout precoce, e costituiscono il gruppo con il **potere discriminante più alto dell'intero dataset** (rapporto Dropout/Graduate ≈ 10:1 al valore 0, documentato in US-07).

**`age_at_enrollment` — studenti adulti come gruppo a rischio, non come rumore.** I valori elevati di `age_at_enrollment` possono superare la soglia dei 3σ. Questi non sono outlier da rimuovere: l'analisi US-07 ha stabilito che questa variabile ha **potere discriminante medio-alto**, con un punto di inversione netto a 22 anni — oltre quella soglia i Dropout prevalgono in modo costante. La coda destra della distribuzione è precisamente il gruppo di studenti adulti che porta informazione unica sul fenomeno dropout.

**Variabili economiche — outlier di coorte, non individuali.** I valori estremi di `unemployment_rate`, `inflation_rate` e `gdp` non sono anomalie individuali: ogni studente condivide lo stesso valore macroeconomico con l'intera coorte del suo anno di iscrizione. L'analisi US-07 ha già evidenziato che `inflation_rate` ha **potere discriminante basso** proprio per questa discretizzazione per coorte (solo 10 valori distinti nel dataset). I loro "outlier" sono valori storici reali, non errori di rilevazione.

### Decisione
Gli outlier identificati con questo metodo vengono documentati ma non rimossi. Saranno confrontati con i risultati degli altri metodi (US-11, US-12, US-13) prima di prendere una decisione finale.

---

## US-11 · Analisi degli Outlier — Metodo Boxplot (IQR) <a id="us-11"></a>

**Responsabile**: Youssra Zarouky

Un valore e outlier se inferiore a `Q1 - 1.5 * IQR` oppure superiore a `Q3 + 1.5 * IQR`.

In [ ]:
numerical_cols = [
    'previous_qualification_grade', 'admission_grade', 'age_at_enrollment',
    'cu_1st_sem_credited', 'cu_1st_sem_enrolled', 'cu_1st_sem_evaluations',
    'cu_1st_sem_approved', 'cu_1st_sem_grade', 'cu_1st_sem_without_evaluations',
    'cu_2nd_sem_credited', 'cu_2nd_sem_enrolled', 'cu_2nd_sem_evaluations',
    'cu_2nd_sem_approved', 'cu_2nd_sem_grade', 'cu_2nd_sem_without_evaluations',
    'unemployment_rate', 'inflation_rate', 'gdp'
]

iqr_summary = detect_outliers_iqr(df, numerical_cols)
std_summary = detect_outliers_std(df, numerical_cols)

comparison = iqr_summary[['colonna', 'n_outlier_iqr']].merge(
    std_summary[['colonna', 'n_outlier']].rename(columns={'n_outlier': 'n_outlier_std'}),
    on='colonna'
).sort_values('n_outlier_iqr', ascending=False)

print('Confronto IQR vs Deviazione Standard:')
print(comparison.to_string(index=False))

In [ ]:
boxplot_vars = [
    'age_at_enrollment', 'admission_grade',
    'cu_1st_sem_approved', 'cu_2nd_sem_approved',
    'cu_1st_sem_grade', 'cu_2nd_sem_grade',
]
target_labels = {0: 'Graduate', 1: 'Dropout'}

fig, axes = plt.subplots(3, 2, figsize=(14, 15))
axes = axes.flatten()

for i, var in enumerate(boxplot_vars):
    plot_df = df[[var, 'target']].copy()
    plot_df['target'] = plot_df['target'].map(target_labels)
    sns.boxplot(data=plot_df, x='target', y=var, palette=['#3498db', '#85c1e9'], ax=axes[i])
    label_it = DEFAULT_DISPLAY_NAMES.get(var, var)
    axes[i].set_title(label_it, fontsize=13, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel(label_it)

plt.suptitle('Boxplot per classe target — variabili chiave (1° e 2° semestre)', fontsize=15, fontweight='bold')
plt.tight_layout()

os.makedirs('../outputs/figures', exist_ok=True)
plt.savefig('../outputs/figures/us11_boxplot_iqr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Boxplot salvati.')

### Osservazioni

Il metodo IQR tende a identificare più outlier rispetto al metodo della deviazione standard, perché è più robusto alle distribuzioni asimmetriche e non assume normalità.

**Lo zero accademico non è un outlier, è il segnale.** Le variabili `cu_*_approved` e `cu_*_grade` mostrano molti outlier IQR nella coda inferiore. Come già discusso in US-07 e US-10, il valore `0` è un placeholder semantico per "nessun esame superato" — non è un voto reale nella fascia bassa. È, anzi, il segnale predittivo più forte del dataset: rimuoverlo significherebbe eliminare la firma principale del dropout precoce.

**Il pattern si rafforza dal 1° al 2° semestre.** I boxplot mostrano le variabili di entrambi i semestri affiancate per confronto diretto. Coerentemente con quanto osservato in US-07, la mediana del gruppo Dropout si abbassa e i whisker si restringono verso 0 passando dal 1° al 2° semestre: la massa Dropout si concentra sempre di più nella fascia di rischio massima nel corso dell'anno. Questa progressione è reale e informativa, non rumore da eliminare.

### Decisione
I risultati verranno inclusi nel confronto finale della US-13 prima di decidere come gestire gli outlier.

---

## US-12 · Analisi degli Outlier — DBSCAN <a id="us-12"></a>

**Responsabile**: Youssra Zarouky

DBSCAN classifica i punti in core, border e noise. I punti noise (etichetta -1) sono i nostri outlier multidimensionali.

In [ ]:
dbscan_cols = ['admission_grade', 'age_at_enrollment', 'cu_1st_sem_approved', 'cu_1st_sem_grade']
X = df[dbscan_cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(X_scaled)

n_noise = (labels == -1).sum()
pct_noise = round((n_noise / len(df)) * 100, 2)
print('Punti noise (outlier): ' + str(n_noise) + ' (' + str(pct_noise) + '%)')
print('Cluster trovati: ' + str(len(set(labels)) - (1 if -1 in labels else 0)))

In [ ]:
noise_df = df[labels == -1][['target']].copy()
target_labels = {0: 'Graduate', 1: 'Dropout'}
noise_df['target'] = noise_df['target'].map(target_labels)
dist = noise_df['target'].value_counts(normalize=True).mul(100).round(2)
print('Distribuzione target tra gli outlier DBSCAN:')
print(dist.to_string())

### Parametri eps e min_samples

- **eps**: raggio del vicinato. Più piccolo = più noise.
- **min_samples**: minimo punti per formare un cluster. Più alto = più conservativo.

### Criteri di selezione delle variabili

Le 4 variabili usate per DBSCAN sono state scelte in base alla classificazione di **potere discriminante** prodotta in US-07:

| Variabile | Potere discriminante (US-07) | Motivazione della selezione |
|---|---|---|
| `cu_1st_sem_approved` | **Alto** | Segnale predittivo più forte del dataset |
| `cu_1st_sem_grade` | **Alto** | Complementare a `_approved`, dimensione qualitativa |
| `age_at_enrollment` | **Medio-alto** | Inversione netta a 22 anni, coda ad alto rischio |
| `admission_grade` | **Medio-basso** | Inclusa per coprire il bagaglio pre-universitario |

Le variabili del 2° semestre (`cu_2nd_sem_*`) non sono state incluse per contenere la dimensionalità di DBSCAN: come documentato in US-07, i pattern del 2° semestre confermano e intensificano quelli del 1°, quindi la loro esclusione non impoverisce l'analisi multidimensionale.

### Osservazioni
DBSCAN individua outlier multidimensionali. Gli outlier tendono ad essere più frequentemente Dropout — coerente con quanto atteso per i profili atipici già identificati nelle US precedenti: studenti con 0 esami superati, voti bassi, età elevata all'iscrizione.

### Decisione
I risultati verranno inclusi nel confronto finale della US-13.

---

## US-13 · Analisi degli Outlier — Isolation Forest e Confronto Finale <a id="us-13"></a>

**Responsabile**: Youssra Zarouky

Isolation Forest costruisce alberi casuali che isolano i punti anomali con pochi tagli. I punti facili da isolare sono outlier.

In [ ]:
numerical_cols = [
    'previous_qualification_grade', 'admission_grade', 'age_at_enrollment',
    'cu_1st_sem_credited', 'cu_1st_sem_enrolled', 'cu_1st_sem_evaluations',
    'cu_1st_sem_approved', 'cu_1st_sem_grade', 'cu_1st_sem_without_evaluations',
    'cu_2nd_sem_credited', 'cu_2nd_sem_enrolled', 'cu_2nd_sem_evaluations',
    'cu_2nd_sem_approved', 'cu_2nd_sem_grade', 'cu_2nd_sem_without_evaluations',
    'unemployment_rate', 'inflation_rate', 'gdp'
]

if_labels = detect_outliers_isolation_forest(df, numerical_cols)
n_if = (if_labels == -1).sum()
pct_if = round((n_if / len(df)) * 100, 2)
print('Outlier Isolation Forest: ' + str(n_if) + ' (' + str(pct_if) + '%)')

In [ ]:
# DBSCAN su 4 variabili
dbscan_cols = ['admission_grade', 'age_at_enrollment', 'cu_1st_sem_approved', 'cu_1st_sem_grade']
X_scaled = StandardScaler().fit_transform(df[dbscan_cols])
dbscan_labels = DBSCAN(eps=0.5, min_samples=5).fit_predict(X_scaled)
n_dbscan = (dbscan_labels == -1).sum()

# Std e IQR
std_summary = detect_outliers_std(df, numerical_cols)
iqr_summary = detect_outliers_iqr(df, numerical_cols)
n_std = std_summary['n_outlier'].sum()
n_iqr = iqr_summary['n_outlier_iqr'].sum()

comparison = pd.DataFrame({
    'Metodo': ['Deviazione Standard (3sigma)', 'IQR (1.5x)', 'DBSCAN', 'Isolation Forest'],
    'N outlier totali': [n_std, n_iqr, n_dbscan, n_if],
    'Note': [
        'Assume distribuzione normale',
        'Robusto alle distribuzioni asimmetriche',
        'Outlier multidimensionali su 4 variabili',
        'Applicato su tutte le variabili numeriche'
    ]
})
print(comparison.to_string(index=False))

### Confronto finale dei quattro metodi

Ogni metodo identifica gli outlier in modo diverso:
- **Deviazione standard**: sensibile alla normalità della distribuzione
- **IQR**: più robusto, identifica più outlier nelle distribuzioni asimmetriche
- **DBSCAN**: rileva anomalie multidimensionali su poche variabili selezionate
- **Isolation Forest**: il più completo, lavora su tutte le variabili numeriche

### Nota sulla ridondanza delle variabili accademiche

Isolation Forest è applicato su tutte le variabili numeriche, incluse le quattro variabili accademiche (`cu_1st_sem_approved`, `cu_1st_sem_grade`, `cu_2nd_sem_approved`, `cu_2nd_sem_grade`). Come segnalato in US-07, queste variabili sono **fortemente correlate** e condividono la stessa semantica al valore 0 (nessun esame superato). Il numero di outlier multivariabili rilevati da Isolation Forest può quindi sovrastimare la rarità di certi profili: uno studente con `0` in tutte e quattro le variabili viene isolato rapidamente non perché sia anomalo nel senso statistico, ma perché lo stesso segnale viene pesato quattro volte. Questo va tenuto a mente nell'interpretare il conteggio totale.

### Decisione finale
Gli outlier **non vengono rimossi**. Le motivazioni sono:

1. **I casi estremi sono i più informativi.** Gli studenti con 0 esami superati, voto medio 0 o età elevata non sono rumore: come documentato in US-07, sono esattamente i gruppi con il potere discriminante più alto del dataset. Rimuoverli impoverirebbe il training set eliminando la firma principale del dropout precoce.

2. **Gli studenti adulti sono un gruppo a rischio specifico.** La coda destra di `age_at_enrollment` non è un errore di rilevazione: l'analisi US-07 ha stabilito che oltre i 22 anni i Dropout prevalgono in modo costante, e questi casi portano informazione unica sul fenomeno.

3. **I modelli che useremo nello Sprint 3 sono robusti agli outlier.** Random Forest e Decision Tree partizionano lo spazio per soglie e non sono sensibili alla scala o alla presenza di valori estremi, a condizione che questi siano informativi — il che è esattamente il caso qui.

La presenza di outlier verrà gestita implicitamente dai modelli di machine learning. La ridondanza delle variabili accademiche (punto critico indipendente) verrà affrontata in Sprint 3 in fase di feature selection.